# DermaVQA — Exploración del Dataset

Análisis exploratorio del dataset DermaVQA (IIYI) para el proyecto de fine-tuning de VLMs en dermatología en español.

**Prerequisito**: copiar `config.yaml.example` → `config.yaml` en la raíz del proyecto y ajustar los paths.

In [ ]:
import os
import json
import re
from pathlib import Path
from collections import Counter

import yaml
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
# Resolve project root (works from notebooks/ or from root)
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

def resolve(p):
    """Resolve path relative to PROJECT_ROOT if not absolute."""
    p = Path(p)
    return p if p.is_absolute() else PROJECT_ROOT / p

config_path = PROJECT_ROOT / "config.yaml"

if os.environ.get("DERMAVQA_DATA_DIR"):
    DATA_DIR   = Path(os.environ["DERMAVQA_DATA_DIR"])
    OUTPUT_DIR = Path(os.environ.get("DERMAVQA_OUTPUT_DIR", PROJECT_ROOT / "outputs"))
elif config_path.exists():
    with open(config_path, "r") as f:
        cfg = yaml.safe_load(f)
    DATA_DIR   = resolve(cfg["data_dir"])
    OUTPUT_DIR = resolve(cfg["output_dir"])
else:
    DATA_DIR   = PROJECT_ROOT / "data" / "iiyi"
    OUTPUT_DIR = PROJECT_ROOT / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_DIR     :", DATA_DIR)
print("DATA_DIR ok  :", DATA_DIR.exists())
print("OUTPUT_DIR   :", OUTPUT_DIR)

## 1. Carga de datos

In [ ]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_data = load_json(DATA_DIR / "train.json")
valid_data = load_json(DATA_DIR / "valid_ht.json")
test_data  = load_json(DATA_DIR / "test_ht.json")

df_train = pd.DataFrame(train_data);  df_train["split"] = "train"
df_valid = pd.DataFrame(valid_data);  df_valid["split"] = "valid"
df_test  = pd.DataFrame(test_data);   df_test["split"]  = "test"

df_all = pd.concat([df_train, df_valid, df_test], ignore_index=True)

print(f"Total casos: {len(df_all)}")
print(df_all["split"].value_counts())

## 2. Visualización de casos

In [ ]:
IMAGES_DIR = DATA_DIR / "images_final"

def find_image_path(image_id):
    matches = list(IMAGES_DIR.rglob(image_id))
    return matches[0] if matches else None

def show_dermavqa_case(df, idx=0, lang="es"):
    row = df.iloc[idx]
    print("=" * 80)
    print("Encounter ID:", row["encounter_id"])
    print("=" * 80)
    print(f"\nPREGUNTA ({lang.upper()})")
    print("-" * 40)
    print(row.get(f"query_title_{lang}", ""))
    print(row.get(f"query_content_{lang}", ""))
    print("\nRESPUESTAS DE REFERENCIA")
    print("-" * 40)
    for i, resp in enumerate(row.get("responses", [])):
        print(f"[{i+1}] {resp.get(f'content_{lang}', '')}")
    print("\nIMAGENES")
    for image_id in row.get("image_ids", []):
        img_path = find_image_path(image_id)
        if img_path is None:
            print("  No encontrada:", image_id)
            continue
        img = Image.open(img_path)
        plt.figure(figsize=(4, 4))
        plt.imshow(img)
        plt.axis("off")
        plt.title(image_id)
        plt.show()

In [ ]:
show_dermavqa_case(df_train, idx=0, lang="es")

## 3. Análisis de respuestas

In [ ]:
def normalize_text(text):
    if text is None:
        return ""
    text = str(text).strip().lower()
    text = re.sub(r"[.,;:]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

def get_answers(responses, lang="es"):
    if not isinstance(responses, list):
        return []
    return [
        str(r[f"content_{lang}"]).strip()
        for r in responses
        if isinstance(r, dict) and r.get(f"content_{lang}")
    ]

def get_majority_answer(responses, lang="es"):
    answers = get_answers(responses, lang)
    if not answers:
        return None
    counts = Counter(normalize_text(a) for a in answers)
    majority_norm = counts.most_common(1)[0][0]
    return next((a for a in answers if normalize_text(a) == majority_norm), majority_norm)

def get_longest_answer(responses, lang="es"):
    answers = get_answers(responses, lang)
    if not answers:
        return None
    return max(answers, key=lambda x: len(x.split()))

def majority_agreement(responses, lang="es"):
    answers = get_answers(responses, lang)
    if not answers:
        return 0.0
    counts = Counter(normalize_text(a) for a in answers)
    return counts.most_common(1)[0][1] / len(answers)

In [ ]:
# Construir campos derivados
for lang in ["en", "es"]:
    df_all[f"question_{lang}"] = (
        df_all[f"query_title_{lang}"].fillna("") + "\n" +
        df_all[f"query_content_{lang}"].fillna("")
    )
    df_all[f"majority_answer_{lang}"]  = df_all["responses"].apply(lambda x: get_majority_answer(x, lang))
    df_all[f"longest_answer_{lang}"]   = df_all["responses"].apply(lambda x: get_longest_answer(x, lang))
    df_all[f"agreement_{lang}"]        = df_all["responses"].apply(lambda x: majority_agreement(x, lang))
    df_all[f"num_answers_{lang}"]      = df_all["responses"].apply(lambda x: len(get_answers(x, lang)))

print("Acuerdo medio (ES):", df_all["agreement_es"].mean().round(3))
print("Acuerdo medio (EN):", df_all["agreement_en"].mean().round(3))
df_all["agreement_es"].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_all["agreement_es"].hist(bins=30, ax=axes[0], edgecolor="black")
axes[0].set_title("Distribución del acuerdo entre respuestas")
axes[0].set_xlabel("Acuerdo (fracción mayoritaria)")
axes[0].set_ylabel("Casos")

df_all["num_answers_es"].hist(bins=20, ax=axes[1], edgecolor="black")
axes[1].set_title("Número de respuestas por caso")
axes[1].set_xlabel("Cantidad de respuestas")
axes[1].set_ylabel("Casos")

plt.tight_layout()
plt.show()

## 4. Construcción del dataset para entrenamiento

Expandimos de casos → pares (imagen, pregunta, respuesta). Usamos la respuesta más larga como target.

In [ ]:
rows = []
for _, row in df_all.iterrows():
    image_ids = row.get("image_ids", [])
    if not isinstance(image_ids, list):
        image_ids = []
    for image_id in image_ids:
        img_path = find_image_path(image_id)
        rows.append({
            "encounter_id"          : row["encounter_id"],
            "split"                 : row["split"],
            "image_id"              : image_id,
            "image_path"            : str(img_path) if img_path else None,
            "question_es"           : row["question_es"],
            "question_en"           : row["question_en"],
            "target_answer_es"      : row["longest_answer_es"],
            "target_answer_en"      : row["longest_answer_en"],
            "agreement_es"          : row["agreement_es"],
            "num_answers"           : row["num_answers_es"],
        })

df_dataset = pd.DataFrame(rows)

print(f"Total filas         : {len(df_dataset)}")
print(f"Imágenes faltantes  : {df_dataset['image_path'].isna().sum()}")
print()
print(df_dataset["split"].value_counts())

In [ ]:
# Longitud de preguntas y respuestas target
df_dataset["q_len_es"]      = df_dataset["question_es"].fillna("").apply(lambda x: len(x.split()))
df_dataset["target_len_es"] = df_dataset["target_answer_es"].fillna("").apply(lambda x: len(x.split()))

print("Longitud media pregunta ES :", df_dataset["q_len_es"].mean().round(1), "palabras")
print("Longitud media respuesta ES:", df_dataset["target_len_es"].mean().round(1), "palabras")
df_dataset["target_len_es"].describe()

## 5. Guardar dataset procesado

In [ ]:
out_path = OUTPUT_DIR / "dermavqa_iiyi_dataset.csv"
df_dataset.to_csv(out_path, index=False, encoding="utf-8")
print("Guardado en:", out_path)